In [ ]:
import torch
import numpy as np
import transformers

transformers.set_seed(42)



In [ ]:
from datasets import load_dataset

dataset = load_dataset("zeroshot/twitter-financial-news-sentiment")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sent_train.csv: 0.00B [00:00, ?B/s]

sent_valid.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/9543 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2388 [00:00<?, ? examples/s]

In [ ]:
dataset.keys()

dict_keys(['train', 'validation'])

In [ ]:
dataset['train'].features['label'].names

AttributeError: 'Value' object has no attribute 'names'

In [ ]:
dataset['train'].features

{'text': Value('string'), 'label': Value('int64')}

In [ ]:
train_set  = dataset['train']
val_set = dataset['validation']

In [ ]:
class_dist = np.unique(train_set['label'], return_counts=True)

In [ ]:
class_weights = torch.FloatTensor(1 / class_dist[1])
class_weights


tensor([0.0007, 0.0005, 0.0002])

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
class_weights = class_weights.to(device)


In [ ]:
from transformers import AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [ ]:
def tokenize(example):
  text = example['text']
  return tokenizer(text, padding="max_length", max_length=128, truncation=True)


encoded_dataset = dataset.map(tokenize)



Map:   0%|          | 0/2388 [00:00<?, ? examples/s]

In [ ]:
encoded_dataset['train'].features

{'text': Value('string'),
 'label': Value('int64'),
 'input_ids': List(Value('int32')),
 'token_type_ids': List(Value('int8')),
 'attention_mask': List(Value('int8'))}

In [ ]:
encoded_dataset = encoded_dataset.remove_columns("text")
encoded_dataset.set_format(type="torch")

In [ ]:
encoded_dataset['train'][1]

{'label': tensor(0),
 'input_ids': tensor([  101,  1002, 10507,  2140,  1002, 22110,  2140,  1011,  2053, 16069,
          2685,  2000, 21725,  2015, 11251,  2012, 11485,  1998,  2548,  7139,
         16770,  1024,  1013,  1013,  1056,  1012,  2522,  1013,  1061,  2290,
          3501, 13876,  2475,  5596,  2509,   102,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,   

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=3)

model = model.to(device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

peft_config = LoraConfig(task_type = TaskType.SEQ_CLS, target_modules=["q_lin", "v_lin"] , r = 8, lora_alpha=16)

peft_model = get_peft_model(model, peft_config)

In [ ]:
peft_model.print_trainable_parameters()

trainable params: 740,355 || all params: 67,696,134 || trainable%: 1.0936


In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(encoded_dataset['train'], batch_size=32, shuffle=True)
val_loader = DataLoader(encoded_dataset['validation'], batch_size=32, shuffle=False)

In [ ]:
from torch import nn

optimizer = torch.optim.AdamW(peft_model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
import evaluate

epochs = 2
loss_score = 0
num_batches = len(train_loader)
val_loss_score = 0

for epoch in range(epochs):
  peft_model.train()
  for batch in train_loader:
    input_id = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    label = batch['label'].to(device)

    # forward pass
    output = peft_model(input_ids = input_id, attention_mask = attention_mask)
    output = output.logits
    # loss
    loss = criterion(output, label)
    # back pass
    loss.backward()
    # adjust weights
    optimizer.step()
    # set grad = 0
    optimizer.zero_grad()
    loss_score += loss.item()
  print(f"Average loss at epoch: {epoch} = {loss_score / num_batches}")
  loss_score = 0

  # Validation loop
  metric = evaluate.load("f1")
  peft_model.eval()
  with torch.no_grad():
    for batch in val_loader:
      input_ids = batch['input_ids'].to(device)
      attention_mask = batch['attention_mask'].to(device)
      label = batch['label'].to(device)

      # evaluate
      output = peft_model(input_ids=input_ids, attention_mask = attention_mask)
      pred_label = torch.argmax(output.logits, dim=1)

      #loss
      loss = criterion(output.logits, label)
      val_loss_score += loss.item()

      # f1 per class
      add_pred = metric.add_batch(predictions=pred_label, references = label)
    f1 = metric.compute(average=None)

    print(f1)
    print(f"Average loss at epoch: {epoch} = {val_loss_score / len(val_loader)}")
    val_loss_score = 0





Average loss at epoch: 0 = 0.5791216164229307
{'f1': array([0.64      , 0.69194313, 0.83232896])}
Average loss at epoch: 0 = 0.5738864385088285
Average loss at epoch: 1 = 0.5561511228515152
{'f1': array([0.65207373, 0.69549218, 0.83445587])}
Average loss at epoch: 1 = 0.5558210735519727


In [ ]:
from huggingface_hub import login
login()

In [ ]:
peft_model.push_to_hub("shakurahmad/finsight-distilbert")
tokenizer.push_to_hub("shakurahmad/finsight-distilbert")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  19%|#8        |  557kB / 2.97MB            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/shakurahmad/finsight-distilbert/commit/96400b0e339b02c83c9e2073e6a93501fbfe3a0d', commit_message='Upload tokenizer', commit_description='', oid='96400b0e339b02c83c9e2073e6a93501fbfe3a0d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/shakurahmad/finsight-distilbert', endpoint='https://huggingface.co', repo_type='model', repo_id='shakurahmad/finsight-distilbert'), pr_revision=None, pr_num=None)